# Does There Reply? - Neural Network Binary Classifier

A frozen BERT encoder creates text features and a configurable MLP predicts labels `0/1`. The supplied `validation.csv` is used for MAE and macro-F1.

In [1]:
from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, mean_absolute_error
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel, AutoTokenizer

SEED = 42
# Hyperparameters.
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
HIDDEN_DIMS = (128, 32)
DROPOUT = 0.20
MAX_EPOCHS = 100
PATIENCE = 15
THRESHOLD = 0.5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'train.csv').exists():
    DATA_DIR = DATA_DIR / 'doesthereply'
train_df = pd.read_csv(DATA_DIR / 'train.csv')
dev_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(f'Train: {len(train_df)} | validation: {len(dev_df)} | test: {len(test_df)}')

c:\Users\bradm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 12 | validation: 4 | test: 8


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert = AutoModel.from_pretrained('bert-base-uncased').to(device).eval()

def embed_texts(texts):
    outputs = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = tokenizer(texts[start:start + BATCH_SIZE], padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs.append(bert(**batch).last_hidden_state[:, 0, :].cpu())
    return torch.cat(outputs).float()

X_train = embed_texts(train_df['text'].fillna('').astype(str).tolist())
X_dev = embed_texts(dev_df['text'].fillna('').astype(str).tolist())
X_test = embed_texts(test_df['text'].fillna('').astype(str).tolist())
print(f'Embedding shape: {tuple(X_train.shape)} | device: {device}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1800.65it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (12, 768) | device: cuda


In [3]:
class ReplyClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]), nn.ReLU(), nn.Dropout(dropout / 2),
            nn.Linear(hidden_dims[1], 1))

    def forward(self, features):
        return self.network(features).squeeze(1)

train_y = torch.tensor(train_df['label'].to_numpy(dtype='float32'))
train_loader = DataLoader(TensorDataset(X_train, train_y), batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))
model = ReplyClassifier(X_train.shape[1], HIDDEN_DIMS, DROPOUT).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
best_weights, best_mae, no_improvement = None, float('inf'), 0

for epoch in range(MAX_EPOCHS):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_x.to(device)), batch_y.to(device))
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        dev_scores = torch.sigmoid(model(X_dev.to(device))).cpu().numpy()
    dev_predictions = (dev_scores >= THRESHOLD).astype(int)
    dev_mae = mean_absolute_error(dev_df['label'], dev_predictions)
    dev_f1 = f1_score(dev_df['label'], dev_predictions, average='macro', zero_division=0)
    if dev_mae < best_mae:
        best_mae, best_weights, no_improvement = dev_mae, copy.deepcopy(model.state_dict()), 0
    else:
        no_improvement += 1
    if (epoch + 1) % 10 == 0 or no_improvement == 0:
        print(f'Epoch {epoch + 1:03d} | validation MAE: {dev_mae:.4f} | macro-F1: {dev_f1:.4f}')
    if no_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch + 1}')
        break

model.load_state_dict(best_weights)
torch.save(model.state_dict(), DATA_DIR / 'best_does_there_reply_model.pt')
print(f'Best validation MAE: {best_mae:.4f}')

Epoch 001 | validation MAE: 0.2500 | macro-F1: 0.4286
Epoch 010 | validation MAE: 0.2500 | macro-F1: 0.4286
Early stopping at epoch 16
Best validation MAE: 0.2500


In [4]:
model.eval()
with torch.no_grad():
    test_scores = torch.sigmoid(model(X_test.to(device))).cpu().numpy()
test_predictions = (test_scores >= THRESHOLD).astype(int)
submission = pd.DataFrame({'id': test_df['id'], 'label': test_predictions})
submission_path = DATA_DIR / 'submission_nn.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} predictions to {submission_path}')
if 'label' in test_df.columns:
    print(f'Test macro-F1: {f1_score(test_df["label"], test_predictions, average="macro", zero_division=0):.4f}')
else:
    print('Test macro-F1 unavailable: test.csv has no ground-truth label column.')
submission.head()

Saved 8 predictions to c:\Users\bradm\Documents\GitHub\ai-masters-workspace\ARI-510-Machine-Learning\Phase 2b\doesthereply\submission_nn.csv
Test macro-F1 unavailable: test.csv has no ground-truth label column.


,id,label
0,DIY_328278_328285,1
1,DIY_328287_328288,1
2,DIY_328293_328297,1
3,DIY_328296_328299,1
4,DIY_328327_328330,1
